In [ ]:
!pip install datasets transformers[torch] evaluate jiwer librosa

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from datasets import Dataset, Audio
import re
import torch
import librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import evaluate
import pandas as pd
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback

# A1. Data Hazırlığı

In [ ]:
DATA_PATH = "/content/drive/MyDrive/azerbaijani/"

TEST_FILE = "test.tsv"
CLIPS_DIR = "clips/"

df_test = pd.read_csv(f"{DATA_PATH}{TEST_FILE}", sep='\t')
df_test['path'] = DATA_PATH + CLIPS_DIR + df_test['path']

test_dataset = Dataset.from_pandas(df_test)
test_dataset = test_dataset.cast_column("path", Audio(sampling_rate=16000))

print(f"Dataset yükləndi. Nümunə sayı: {len(test_dataset)}")

Dataset yükləndi. Nümunə sayı: 126


In [ ]:
def normalize_text(text):

    text = text.lower()

    text = re.sub(r'[^\w\s]', '', text)

    text = " ".join(text.split())
    return text

# A2. Model Seçimi və İnferens

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_ID = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(device)

sample_path = df_test['path'].iloc[0]
speech, _ = librosa.load(sample_path, sr=16000)

input_features = processor(speech, sampling_rate=16000, return_tensors="pt").input_features.to(device)
predicted_ids = model.generate(input_features, language="azerbaijani")
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)

print(f"Orijinal Mətn: {df_test['sentence'].iloc[0]}")
print(f"Modelin Təxmini: {transcription[0]}")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

Orijinal Mətn: Eşq şeirlərindən əlavə din və təsəvvüf ilə bağlı şeirləri də vardır.
Modelin Təxmini:  Eşk şəirlərindən ələvə din vətə sə vüf ilə bağlı şeirləri dəvardır.


In [ ]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

In [ ]:
predictions = []
references = []

for i in range(len(df_test)):

    path = df_test['path'].iloc[i]
    speech, _ = librosa.load(path, sr=16000)

    input_features = processor(speech, sampling_rate=16000, return_tensors="pt").input_features.to(device)

    with torch.no_grad():
        predicted_ids = model.generate(input_features, language="azerbaijani")

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    predictions.append(transcription)
    references.append(df_test['sentence'].iloc[i])

print("İnferens tamamlandı.")

İnferens tamamlandı.


# A3. Performans Qiymətləndirməsi

In [ ]:
overall_wer = wer_metric.compute(predictions=predictions, references=references)
overall_cer = cer_metric.compute(predictions=predictions, references=references)

print(f"\nOrtalama WER: {overall_wer * 100:.2f}%")
print(f"Ortalama CER: {overall_cer * 100:.2f}%")


Ortalama WER: 64.94%
Ortalama CER: 20.82%


In [ ]:
results_a3 = []

for i in range(len(predictions)):
    wer_val = wer_metric.compute(predictions=[predictions[i]], references=[references[i]])

    results_a3.append({
        "Reference (Orijinal)": references[i],
        "Prediction (Model)": predictions[i],
        "WER (%)": round(wer_val * 100, 2)
    })
df_a3 = pd.DataFrame(results_a3)

best_5_a3 = df_a3.sort_values(by="WER (%)").head(5)


worst_5_a3 = df_a3.sort_values(by="WER (%)", ascending=False).head(5)

print("--- HİSSƏ A3: BAZA MODELİN ƏN YAXŞI 5 NÜMÜNƏSİ ---")
print(best_5_a3.to_string(index=False))

print("\n--- HİSSƏ A3: BAZA MODELİN ƏN PİS 5 NÜMÜNƏSİ ---")
print(worst_5_a3.to_string(index=False))

--- HİSSƏ A3: BAZA MODELİN ƏN YAXŞI 5 NÜMÜNƏSİ ---
                                         Reference (Orijinal)                                             Prediction (Model)  WER (%)
                                            Adada göl yoxdur.                                              Adada göl yoxdur.     0.00
                                              Çünki o yoxdur.                                                Çünki o yoxdur.     0.00
                    Daha sonra həmin məktəbin direktoru olur.                      Daha sonra həmin məcdəbin direktoru olur.    16.67
         Həmin il Yaponiya Misirin müstəqilliyini tanımışdır.            Həmin il Yaponiya Misirin müstəqiliyini tanımışdır.    16.67
Hal hazırda Xəzər kanalında prodüser olaraq işinə davam edir.  Hal hazırda xəzər kanalında producer olaraq işinə davam edir.    22.22

--- HİSSƏ A3: BAZA MODELİN ƏN PİS 5 NÜMÜNƏSİ ---
                                                         Reference (Orijinal)                  